In [58]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Joining Land Registry Price Data with Energy Performance Certificates (2015 - 2025)

In [59]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [60]:
from src.cleaning import load
from src.cleaning import config

In [61]:

pp = load.read_price_paid(config.DATA_PATH, config.PP_FILENAME)
pp.shape[0]

135557

In [62]:
epc = load.read_epc(config.DATA_PATH, config.EPC_FILENAME, config.EPC_COLS)
epc.info()

<class 'pandas.DataFrame'>
RangeIndex: 216054 entries, 0 to 216053
Data columns (total 28 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   certificate_number           216054 non-null  str           
 1   address1                     216054 non-null  str           
 2   address2                     117532 non-null  str           
 3   address3                     27304 non-null   str           
 4   postcode                     216054 non-null  str           
 5   posttown                     213327 non-null  str           
 6   address                      216054 non-null  str           
 7   local_authority              216054 non-null  str           
 8   local_authority_label        216054 non-null  str           
 9   built_form                   213548 non-null  str           
 10  construction_age_band        216054 non-null  str           
 11  current_energy_efficiency    216054 n

Correct row number and dtypes

In [63]:
epc = load.filter_local_authorities(epc, config.TARGET_LOCAL_AUTHORITIES)

epc.shape

(216050, 28)

4 rows dropped as expected

In [64]:
from src.cleaning import normalise

epc = normalise.strip_whitespace(epc)
epc = normalise.remove_commas(epc, config.EPC_ADDRESS_COLS)
epc['address1']

0                  10 Newman Road
1                 10 Barlow Close
2                65 Campbell Road
3           12 St. Michaels Close
4         Flat 10 Canalside House
                   ...           
216049               30 Appletons
216050        1d Walton Well Road
216051            8 Sycamore Road
216052        68 Spencer Crescent
216053             21 Ochre Close
Name: address1, Length: 216050, dtype: str

In [65]:
epc = normalise.build_address(epc, config.EPC_ADDRESS_COLS)

In [66]:
epc = normalise.build_tokens(epc)

epc['address_tokens']

0                   frozenset({ROAD, 3UJ, 10, OX4, NEWMAN})
1         frozenset({CLOSE, WHEATLEY, BARLOW, 10, 1NL, O...
2                 frozenset({ROAD, 3PG, 65, OX4, CAMPBELL})
3         frozenset({MICHAELS, CLOSE, ST., 6BE, SHIPTON-...
4         frozenset({ROAD, TRAMWAY, FLAT, 10, HOUSE, 5BG...
                                ...                        
216049                frozenset({7GG, OX12, 30, APPLETONS})
216050        frozenset({6ED, ROAD, WALTON, WELL, OX2, 1D})
216051             frozenset({ROAD, OX2, 9EJ, 8, SYCAMORE})
216052         frozenset({68, CRESCENT, SPENCER, OX4, 4SW})
216053    frozenset({CLOSE, OCHRE, WHEATLEY, 1FQ, OX33, ...
Name: address_tokens, Length: 216050, dtype: object

In [67]:
pp = normalise.strip_whitespace(pp)
pp = normalise.remove_commas(pp, config.PP_ADDRESS_COLS)
pp = normalise.build_address(pp, config.PP_ADDRESS_COLS)
pp = normalise.build_tokens(pp)

In [68]:
from src.cleaning import match

In [69]:
df = match.merge_on_postcode(pp, epc)
df.shape

(2891332, 46)

In [70]:
df = match.filter_subset_matches(df)
print(df.shape)
print(df['unique_id'].nunique())

(121525, 46)
105419


In [71]:
df = match.drop_ambiguous_matches(df)
print(df.shape)
print(df['unique_id'].nunique())

(117362, 46)
104444


In [83]:
matches = df.copy()

matches['inspection_to_sale'] = matches['deed_date'] - matches['inspection_date']
drop_neg = matches[matches['inspection_to_sale'] >= '0 days']
grouped = drop_neg.groupby('unique_id')['inspection_to_sale'].transform('min') # minimum epc inspection_to_sale for each sale, transform means that the same index is kept
best = drop_neg[drop_neg['inspection_to_sale'] == grouped] # filtered sales matches to those that equal the min time_to_inspection (inc ties)
print(best[best.duplicated(subset='unique_id', keep=False)]['unique_id'].nunique())
out = drop_neg.sort_values(by=['inspection_to_sale', 'lodgement_date'], ascending=[True, False]).drop_duplicates(subset='unique_id')
print(out.shape)

670
(91150, 47)


Ties found: 670 sales (0.7%) had two or more certificates tied for closest pre-sale inspection; previously chosen arbitrarily, now resolved by latest lodgement_date. EPC features for these sales may differ from Part 4.

In [ ]:
df = match.select_presale_epc(df)
df.shape


(91150, 47)

In [56]:
df = match.select_final_columns(df, config.FINAL_COLS, config.RENAME_MAP)
df.shape

(91150, 36)